In [1]:
#Import library
import pandas as pd
import re
import string
import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [2]:
#Baca dataset

df = pd.read_csv("dataset.csv")

text_column = "comment"

print("Ukuran dataset:", df.shape)
print("Daftar kolom:", df.columns.tolist())

df[[text_column]].head(10)


Ukuran dataset: (220051, 8)
Daftar kolom: ['video_id', 'comment_id', 'parent_comment_id', 'level', 'username', 'nickname', 'comment', 'create_time']


,comment
0,Beliau bilang untung nya 2 m ?
1,Makasih Mas
2,Gimana untung nya 2 m..barang yg di jual ajh g...
3,[Sticker] dibilang Makasih Mas udah 2M itu
4,😳😳😳
5,"Kalo niat membantu UMKM, harusnya koprasi mera..."
6,sabar mbak paling 6 bulan dah tutup
7,"kalo ternyata harga murah, saya harap bisa ber..."
8,"SAYA SETUJU BANG, soalnya saya punya warung 😔"
9,Ancen Dancuk 🥵🥵🥵


In [3]:
#Case folding

def case_folding(text):
    return str(text).lower()

df["lowercase_text"] = df[text_column].fillna("").apply(case_folding)

df[[text_column, "lowercase_text"]].head(10)


,comment,lowercase_text
0,Beliau bilang untung nya 2 m ?,beliau bilang untung nya 2 m ?
1,Makasih Mas,makasih mas
2,Gimana untung nya 2 m..barang yg di jual ajh g...,gimana untung nya 2 m..barang yg di jual ajh g...
3,[Sticker] dibilang Makasih Mas udah 2M itu,[sticker] dibilang makasih mas udah 2m itu
4,😳😳😳,😳😳😳
5,"Kalo niat membantu UMKM, harusnya koprasi mera...","kalo niat membantu umkm, harusnya koprasi mera..."
6,sabar mbak paling 6 bulan dah tutup,sabar mbak paling 6 bulan dah tutup
7,"kalo ternyata harga murah, saya harap bisa ber...","kalo ternyata harga murah, saya harap bisa ber..."
8,"SAYA SETUJU BANG, soalnya saya punya warung 😔","saya setuju bang, soalnya saya punya warung 😔"
9,Ancen Dancuk 🥵🥵🥵,ancen dancuk 🥵🥵🥵


In [14]:
# Cleaning teks

def remove_url(text):
    return re.sub(r"https?://\S+|www\.\S+", " ", text)

def remove_mention(text):
    return re.sub(r"@\w+", " ", text)

def remove_hashtag(text):
    return re.sub(r"#\w+", " ", text)

def remove_number(text):
    return re.sub(r"\d+", " ", text)

def remove_punctuation(text):
    return re.sub(r"[^\w\s]", " ", text)

def clean_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

def clean_text(text):
    text = case_folding(text)
    text = remove_url(text)
    text = remove_mention(text)
    text = remove_hashtag(text)
    text = remove_number(text)
    text = remove_punctuation(text)
    text = clean_whitespace(text)
    return text

df["clean_text"] = df[text_column].fillna("").apply(clean_text)

display(df[[text_column, "clean_text"]].head(10))


,comment,clean_text
0,Beliau bilang untung nya 2 m ?,beliau bilang untung nya m
1,Makasih Mas,makasih mas
2,Gimana untung nya 2 m..barang yg di jual ajh g...,gimana untung nya m barang yg di jual ajh ga a...
3,[Sticker] dibilang Makasih Mas udah 2M itu,sticker dibilang makasih mas udah m itu
4,😳😳😳,
5,"Kalo niat membantu UMKM, harusnya koprasi mera...",kalo niat membantu umkm harusnya koprasi merah...
6,sabar mbak paling 6 bulan dah tutup,sabar mbak paling bulan dah tutup
7,"kalo ternyata harga murah, saya harap bisa ber...",kalo ternyata harga murah saya harap bisa berj...
8,"SAYA SETUJU BANG, soalnya saya punya warung 😔",saya setuju bang soalnya saya punya warung
9,Ancen Dancuk 🥵🥵🥵,ancen dancuk


In [5]:
#Tokenisasi

def tokenize_text(text):
    if not text:
        return []
    return word_tokenize(text)

df["tokens"] = df["clean_text"].apply(tokenize_text)

df[["clean_text", "tokens"]].head(10)


,clean_text,tokens
0,beliau bilang untung nya m,"[beliau, bilang, untung, nya, m]"
1,makasih mas,"[makasih, mas]"
2,gimana untung nya mbarang yg di jual ajh ga ad...,"[gimana, untung, nya, mbarang, yg, di, jual, a..."
3,sticker dibilang makasih mas udah m itu,"[sticker, dibilang, makasih, mas, udah, m, itu]"
4,😳😳😳,[😳😳😳]
5,kalo niat membantu umkm harusnya koprasi merah...,"[kalo, niat, membantu, umkm, harusnya, koprasi..."
6,sabar mbak paling bulan dah tutup,"[sabar, mbak, paling, bulan, dah, tutup]"
7,kalo ternyata harga murah saya harap bisa berj...,"[kalo, ternyata, harga, murah, saya, harap, bi..."
8,saya setuju bang soalnya saya punya warung 😔,"[saya, setuju, bang, soalnya, saya, punya, war..."
9,ancen dancuk 🥵🥵🥵,"[ancen, dancuk, 🥵🥵🥵]"


In [6]:
#Stopword removal

stop_words = set(stopwords.words("indonesian"))

# Pertahankan kata negasi agar makna tidak berubah.
for kata in ["tidak", "bukan", "belum", "jangan", "kurang", "tak"]:
    stop_words.discard(kata)

def remove_stopwords(tokens):
    return [token for token in tokens if token not in stop_words]

df["tokens_no_stopword"] = df["tokens"].apply(remove_stopwords)

df[["tokens", "tokens_no_stopword"]].head(10)


,tokens,tokens_no_stopword
0,"[beliau, bilang, untung, nya, m]","[beliau, bilang, untung, nya, m]"
1,"[makasih, mas]","[makasih, mas]"
2,"[gimana, untung, nya, mbarang, yg, di, jual, a...","[gimana, untung, nya, mbarang, yg, jual, ajh, ..."
3,"[sticker, dibilang, makasih, mas, udah, m, itu]","[sticker, dibilang, makasih, mas, udah, m]"
4,[😳😳😳],[😳😳😳]
5,"[kalo, niat, membantu, umkm, harusnya, koprasi...","[kalo, niat, membantu, umkm, koprasi, merah, p..."
6,"[sabar, mbak, paling, bulan, dah, tutup]","[sabar, mbak, dah, tutup]"
7,"[kalo, ternyata, harga, murah, saya, harap, bi...","[kalo, harga, murah, harap, berjalan]"
8,"[saya, setuju, bang, soalnya, saya, punya, war...","[setuju, bang, warung, 😔]"
9,"[ancen, dancuk, 🥵🥵🥵]","[ancen, dancuk, 🥵🥵🥵]"


In [10]:
#Stemming dengan Sastrawi pada Komentar TikTok

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stem_tokens(tokens):
    return [stemmer.stem(token) for token in tokens]

# Uji stemming pada 20 komentar TikTok pertama
uji_stemming = df["tokens_no_stopword"].head(20).apply(stem_tokens)

hasil_uji = pd.DataFrame({
    "Komentar Asli": df["comment"].head(20),
    "Sebelum Stemming": df["tokens_no_stopword"].head(20),
    "Setelah Stemming": uji_stemming
})

display(hasil_uji)

,Komentar Asli,Sebelum Stemming,Setelah Stemming
0,Beliau bilang untung nya 2 m ?,"[beliau, bilang, untung, nya, m]","[beliau, bilang, untung, nya, m]"
1,Makasih Mas,"[makasih, mas]","[makasih, mas]"
2,Gimana untung nya 2 m..barang yg di jual ajh g...,"[gimana, untung, nya, mbarang, yg, jual, ajh, ...","[gimana, untung, nya, mbarang, yg, jual, ajh, ..."
3,[Sticker] dibilang Makasih Mas udah 2M itu,"[sticker, dibilang, makasih, mas, udah, m]","[sticker, bilang, makasih, mas, udah, m]"
4,😳😳😳,[😳😳😳],[]
5,"Kalo niat membantu UMKM, harusnya koprasi mera...","[kalo, niat, membantu, umkm, koprasi, merah, p...","[kalo, niat, bantu, umkm, koprasi, merah, puti..."
6,sabar mbak paling 6 bulan dah tutup,"[sabar, mbak, dah, tutup]","[sabar, mbak, dah, tutup]"
7,"kalo ternyata harga murah, saya harap bisa ber...","[kalo, harga, murah, harap, berjalan]","[kalo, harga, murah, harap, jalan]"
8,"SAYA SETUJU BANG, soalnya saya punya warung 😔","[setuju, bang, warung, 😔]","[tuju, bang, warung, ]"
9,Ancen Dancuk 🥵🥵🥵,"[ancen, dancuk, 🥵🥵🥵]","[ancen, dancuk, ]"


In [13]:
# Contoh Stemming dari 10 Komentar TikTok

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Mengambil 10 komentar dari dataset sendiri
komentar = df["comment"].dropna().astype(str).head(10)

for i, kalimat in enumerate(komentar, start=1):
    print(f"Contoh {i}")
    print("Input :", kalimat)
    print("Output:", stemmer.stem(kalimat))
    print("-" * 50)


Contoh 1
Input : Beliau bilang untung nya 2 m ?
Output: beliau bilang untung nya 2 m
--------------------------------------------------
Contoh 2
Input : Makasih Mas
Output: makasih mas
--------------------------------------------------
Contoh 3
Input : Gimana untung nya 2 m..barang yg di jual ajh ga ada 1 m 😁
Output: gimana untung nya 2 m barang yg di jual ajh ga ada 1 m
--------------------------------------------------
Contoh 4
Input : [Sticker] dibilang Makasih Mas udah 2M itu
Output: sticker bilang makasih mas udah 2m itu
--------------------------------------------------
Contoh 5
Input : 😳😳😳
Output: 
--------------------------------------------------
Contoh 6
Input : Kalo niat membantu UMKM, harusnya koprasi merah putih dibuka dalam bentuk warehouse (gudang), bukan bentuk toko, jadi yang belanja cuma UMKM yang memiliki usaha toko kelontong supaya dapat harga lebih murah dan konsumen toko kelontong dapat harga yang stabil, kalok di buka dalam bentuk toko, dimana semua masyarakat um

In [16]:
# Pipeline Preprocessing

def preprocessing(text):
    text = clean_text(text)
    tokens = tokenize_text(text)
    tokens = remove_stopwords(tokens)
    tokens = stem_tokens(tokens)
    return tokens

# Uji pipeline pada 20 komentar pertama
df["tokens_preprocessed"] = None

df.loc[:19, "tokens_preprocessed"] = (
    df["comment"].fillna("").head(20).apply(preprocessing)
)

display(
    df[["comment", "tokens_preprocessed"]].head(20)
)


,comment,tokens_preprocessed
0,Beliau bilang untung nya 2 m ?,"[beliau, bilang, untung, nya, m]"
1,Makasih Mas,"[makasih, mas]"
2,Gimana untung nya 2 m..barang yg di jual ajh g...,"[gimana, untung, nya, m, barang, yg, jual, ajh..."
3,[Sticker] dibilang Makasih Mas udah 2M itu,"[sticker, bilang, makasih, mas, udah, m]"
4,😳😳😳,[]
5,"Kalo niat membantu UMKM, harusnya koprasi mera...","[kalo, niat, bantu, umkm, koprasi, merah, puti..."
6,sabar mbak paling 6 bulan dah tutup,"[sabar, mbak, dah, tutup]"
7,"kalo ternyata harga murah, saya harap bisa ber...","[kalo, harga, murah, harap, jalan]"
8,"SAYA SETUJU BANG, soalnya saya punya warung 😔","[tuju, bang, warung]"
9,Ancen Dancuk 🥵🥵🥵,"[ancen, dancuk]"


In [17]:
# Menggabungkan Token Menjadi Teks

df["processed_text"] = None

df.loc[:19, "processed_text"] = (
    df.loc[:19, "tokens_preprocessed"].apply(
        lambda tokens: " ".join(tokens)
    )
)

display(
    df[["comment", "tokens_preprocessed", "processed_text"]].head(20)
)


,comment,tokens_preprocessed,processed_text
0,Beliau bilang untung nya 2 m ?,"[beliau, bilang, untung, nya, m]",beliau bilang untung nya m
1,Makasih Mas,"[makasih, mas]",makasih mas
2,Gimana untung nya 2 m..barang yg di jual ajh g...,"[gimana, untung, nya, m, barang, yg, jual, ajh...",gimana untung nya m barang yg jual ajh ga m
3,[Sticker] dibilang Makasih Mas udah 2M itu,"[sticker, bilang, makasih, mas, udah, m]",sticker bilang makasih mas udah m
4,😳😳😳,[],
5,"Kalo niat membantu UMKM, harusnya koprasi mera...","[kalo, niat, bantu, umkm, koprasi, merah, puti...",kalo niat bantu umkm koprasi merah putih buka ...
6,sabar mbak paling 6 bulan dah tutup,"[sabar, mbak, dah, tutup]",sabar mbak dah tutup
7,"kalo ternyata harga murah, saya harap bisa ber...","[kalo, harga, murah, harap, jalan]",kalo harga murah harap jalan
8,"SAYA SETUJU BANG, soalnya saya punya warung 😔","[tuju, bang, warung]",tuju bang warung
9,Ancen Dancuk 🥵🥵🥵,"[ancen, dancuk]",ancen dancuk


In [18]:
# Tabel Perbandingan Hasil Text Normalization
jumlah_contoh = 20

tabel_perbandingan = pd.DataFrame({
    "Komentar Asli": df["comment"].head(jumlah_contoh),
    "Lowercase": df["lowercase_text"].head(jumlah_contoh),
    "Cleaned": df["clean_text"].head(jumlah_contoh),
    "Tokenisasi": df["tokens"].head(jumlah_contoh),
    "Stopword Removal": df["tokens_no_stopword"].head(jumlah_contoh),
    "Setelah Stemming": df["tokens_preprocessed"].head(jumlah_contoh)
})

display(tabel_perbandingan)


,Komentar Asli,Lowercase,Cleaned,Tokenisasi,Stopword Removal,Setelah Stemming
0,Beliau bilang untung nya 2 m ?,beliau bilang untung nya 2 m ?,beliau bilang untung nya m,"[beliau, bilang, untung, nya, m]","[beliau, bilang, untung, nya, m]","[beliau, bilang, untung, nya, m]"
1,Makasih Mas,makasih mas,makasih mas,"[makasih, mas]","[makasih, mas]","[makasih, mas]"
2,Gimana untung nya 2 m..barang yg di jual ajh g...,gimana untung nya 2 m..barang yg di jual ajh g...,gimana untung nya m barang yg di jual ajh ga a...,"[gimana, untung, nya, mbarang, yg, di, jual, a...","[gimana, untung, nya, mbarang, yg, jual, ajh, ...","[gimana, untung, nya, m, barang, yg, jual, ajh..."
3,[Sticker] dibilang Makasih Mas udah 2M itu,[sticker] dibilang makasih mas udah 2m itu,sticker dibilang makasih mas udah m itu,"[sticker, dibilang, makasih, mas, udah, m, itu]","[sticker, dibilang, makasih, mas, udah, m]","[sticker, bilang, makasih, mas, udah, m]"
4,😳😳😳,😳😳😳,,[😳😳😳],[😳😳😳],[]
5,"Kalo niat membantu UMKM, harusnya koprasi mera...","kalo niat membantu umkm, harusnya koprasi mera...",kalo niat membantu umkm harusnya koprasi merah...,"[kalo, niat, membantu, umkm, harusnya, koprasi...","[kalo, niat, membantu, umkm, koprasi, merah, p...","[kalo, niat, bantu, umkm, koprasi, merah, puti..."
6,sabar mbak paling 6 bulan dah tutup,sabar mbak paling 6 bulan dah tutup,sabar mbak paling bulan dah tutup,"[sabar, mbak, paling, bulan, dah, tutup]","[sabar, mbak, dah, tutup]","[sabar, mbak, dah, tutup]"
7,"kalo ternyata harga murah, saya harap bisa ber...","kalo ternyata harga murah, saya harap bisa ber...",kalo ternyata harga murah saya harap bisa berj...,"[kalo, ternyata, harga, murah, saya, harap, bi...","[kalo, harga, murah, harap, berjalan]","[kalo, harga, murah, harap, jalan]"
8,"SAYA SETUJU BANG, soalnya saya punya warung 😔","saya setuju bang, soalnya saya punya warung 😔",saya setuju bang soalnya saya punya warung,"[saya, setuju, bang, soalnya, saya, punya, war...","[setuju, bang, warung, 😔]","[tuju, bang, warung]"
9,Ancen Dancuk 🥵🥵🥵,ancen dancuk 🥵🥵🥵,ancen dancuk,"[ancen, dancuk, 🥵🥵🥵]","[ancen, dancuk, 🥵🥵🥵]","[ancen, dancuk]"


In [20]:
# Menampilkan Perbandingan Text Normalization per Komentar
for i in range(20):
    print(f"\n===== KOMENTAR {i + 1} =====")
    print("Komentar Asli      :", tabel_perbandingan.iloc[i]["Komentar Asli"])
    print("Lowercase          :", tabel_perbandingan.iloc[i]["Lowercase"])
    print("Cleaned            :", tabel_perbandingan.iloc[i]["Cleaned"])
    print("Tokenisasi         :", tabel_perbandingan.iloc[i]["Tokenisasi"])
    print("Stopword Removal   :", tabel_perbandingan.iloc[i]["Stopword Removal"])
    print("Setelah Stemming   :", tabel_perbandingan.iloc[i]["Setelah Stemming"])



===== KOMENTAR 1 =====
Komentar Asli      : Beliau bilang untung nya 2 m ?
Lowercase          : beliau bilang untung nya 2 m ?
Cleaned            : beliau bilang untung nya m
Tokenisasi         : ['beliau', 'bilang', 'untung', 'nya', 'm']
Stopword Removal   : ['beliau', 'bilang', 'untung', 'nya', 'm']
Setelah Stemming   : ['beliau', 'bilang', 'untung', 'nya', 'm']

===== KOMENTAR 2 =====
Komentar Asli      : Makasih Mas
Lowercase          : makasih mas
Cleaned            : makasih mas
Tokenisasi         : ['makasih', 'mas']
Stopword Removal   : ['makasih', 'mas']
Setelah Stemming   : ['makasih', 'mas']

===== KOMENTAR 3 =====
Komentar Asli      : Gimana untung nya 2 m..barang yg di jual ajh ga ada 1 m 😁
Lowercase          : gimana untung nya 2 m..barang yg di jual ajh ga ada 1 m 😁
Cleaned            : gimana untung nya m barang yg di jual ajh ga ada m
Tokenisasi         : ['gimana', 'untung', 'nya', 'mbarang', 'yg', 'di', 'jual', 'ajh', 'ga', 'ada', 'm', '😁']
Stopword Removal   : ['gi

In [21]:

print("Jumlah contoh:", len(tabel_perbandingan))
print("Jumlah data setelah stemming:", 
      tabel_perbandingan["Setelah Stemming"].notna().sum())

display(tabel_perbandingan.tail(5))


Jumlah contoh: 20
Jumlah data setelah stemming: 20


,Komentar Asli,Lowercase,Cleaned,Tokenisasi,Stopword Removal,Setelah Stemming
15,ini saya paling setujuuu,ini saya paling setujuuu,ini saya paling setujuuu,"[ini, saya, paling, setujuuu]",[setujuuu],[setujuuu]
16,"baru mau ngetik gini, ehh udah di duluin","baru mau ngetik gini, ehh udah di duluin",baru mau ngetik gini ehh udah di duluin,"[baru, mau, ngetik, gini, ehh, udah, di, duluin]","[ngetik, gini, ehh, udah, duluin]","[ngetik, gin, ehh, udah, duluin]"
17,"ya ngga buka warehouse juga, apa bedanya, sekarang yang mati justru penjual grosir, distributor ...","ya ngga buka warehouse juga, apa bedanya, sekarang yang mati justru penjual grosir, distributor ...",ya ngga buka warehouse juga apa bedanya sekarang yang mati justru penjual grosir distributor dan...,"[ya, ngga, buka, warehouse, juga, apa, bedanya, sekarang, yang, mati, justru, penjual, grosir, d...","[ya, ngga, buka, warehouse, bedanya, mati, penjual, grosir, distributor, sales, ”, toko, ya, jag...","[ya, ngga, buka, warehouse, beda, mati, jual, grosir, distributor, sales, toko, ya, jaga, harga,..."
18,[Sticker],[sticker],sticker,[sticker],[sticker],[sticker]
19,umkm nya ntar ngambil untung nya keterlaluan 🤣🤣\nsma aja,umkm nya ntar ngambil untung nya keterlaluan 🤣🤣\nsma aja,umkm nya ntar ngambil untung nya keterlaluan sma aja,"[umkm, nya, ntar, ngambil, untung, nya, keterlaluan, 🤣🤣, sma, aja]","[umkm, nya, ntar, ngambil, untung, nya, 🤣🤣, sma, aja]","[umkm, nya, ntar, ngambil, untung, nya, sma, aja]"


In [ ]:
# Uji preprocessing bertahap: 1.000 komentar pertama

jumlah_uji = min(1000, len(df))

hasil_tokens = df["comment"].fillna("").head(jumlah_uji).apply(preprocessing)

df.loc[df.index[:jumlah_uji], "tokens_preprocessed"] = hasil_tokens

df.loc[df.index[:jumlah_uji], "processed_text"] = hasil_tokens.apply(
    lambda tokens: " ".join(tokens)
)

print("Jumlah komentar yang diproses:", jumlah_uji)
print("Total komentar yang memiliki hasil:",
      df["tokens_preprocessed"].notna().sum())

display(df[["comment", "processed_text"]].head(10))


Jumlah komentar yang diproses: 1000
Total komentar yang memiliki hasil: 1000


,comment,processed_text
0,Beliau bilang untung nya 2 m ?,beliau bilang untung nya m
1,Makasih Mas,makasih mas
2,Gimana untung nya 2 m..barang yg di jual ajh ga ada 1 m 😁,gimana untung nya m barang yg jual ajh ga m
3,[Sticker] dibilang Makasih Mas udah 2M itu,sticker bilang makasih mas udah m
4,😳😳😳,
5,"Kalo niat membantu UMKM, harusnya koprasi merah putih dibuka dalam bentuk warehouse (gudang), bu...",kalo niat bantu umkm koprasi merah putih buka bentuk warehouse gudang bukan bentuk toko belanja ...
6,sabar mbak paling 6 bulan dah tutup,sabar mbak dah tutup
7,"kalo ternyata harga murah, saya harap bisa berjalan terus.",kalo harga murah harap jalan
8,"SAYA SETUJU BANG, soalnya saya punya warung 😔",tuju bang warung
9,Ancen Dancuk 🥵🥵🥵,ancen dancuk


In [24]:
# Melanjutkan preprocessing dari komentar ke-1.001
ukuran_batch = 1000
total_data = len(df)

for mulai in range(1000, total_data, ukuran_batch):
    akhir = min(mulai + ukuran_batch, total_data)

    hasil = df["comment"].iloc[mulai:akhir].fillna("").apply(preprocessing)

    df.loc[df.index[mulai:akhir], "tokens_preprocessed"] = hasil
    df.loc[df.index[mulai:akhir], "processed_text"] = hasil.apply(
        lambda tokens: " ".join(tokens)
    )

    print(f"Selesai memproses {akhir} dari {total_data} komentar")

print("Semua komentar selesai diproses.")


Selesai memproses 2000 dari 220051 komentar


KeyboardInterrupt: 

In [25]:

print("Total data:", len(df))
print("Komentar yang sudah diproses:",
      df["tokens_preprocessed"].notna().sum())
print("Hasil teks yang sudah tersedia:",
      df["processed_text"].notna().sum())


Total data: 220051
Komentar yang sudah diproses: 2000
Hasil teks yang sudah tersedia: 2000


In [26]:
# Cache untuk menyimpan hasil stemming setiap kata
cache_stemming = {}

def stem_tokens(tokens):
    hasil = []

    for token in tokens:
        if token not in cache_stemming:
            cache_stemming[token] = stemmer.stem(token)

        hasil.append(cache_stemming[token])

    return hasil

print("Fungsi stemming dengan cache siap digunakan.")


Fungsi stemming dengan cache siap digunakan.


In [27]:
import time

# Uji pada 100 komentar yang belum diproses
mulai_uji = 2000
akhir_uji = min(mulai_uji + 100, len(df))

waktu_mulai = time.time()

hasil_uji = (
    df["comment"]
    .iloc[mulai_uji:akhir_uji]
    .fillna("")
    .apply(preprocessing)
)

waktu_selesai = time.time() - waktu_mulai

print("Komentar yang diuji:", len(hasil_uji))
print("Waktu pengujian:", round(waktu_selesai, 2), "detik")
print("Jumlah kata dalam cache:", len(cache_stemming))


Komentar yang diuji: 100
Waktu pengujian: 0.06 detik
Jumlah kata dalam cache: 514


In [28]:
import time

ukuran_batch = 5000
total_data = len(df)
waktu_mulai_total = time.time()

for mulai in range(2000, total_data, ukuran_batch):
    akhir = min(mulai + ukuran_batch, total_data)

    hasil = (
        df["comment"]
        .iloc[mulai:akhir]
        .fillna("")
        .apply(preprocessing)
    )

    df.loc[df.index[mulai:akhir], "tokens_preprocessed"] = hasil
    df.loc[df.index[mulai:akhir], "processed_text"] = hasil.apply(
        lambda tokens: " ".join(tokens)
    )

    print(f"Selesai memproses {akhir} dari {total_data} komentar")

print("Semua komentar selesai diproses.")
print("Total waktu:", round(time.time() - waktu_mulai_total, 2), "detik")


Selesai memproses 7000 dari 220051 komentar


KeyboardInterrupt: 

In [29]:
print("Total dataset:", len(df))
print("Komentar sudah diproses:",
      df["tokens_preprocessed"].notna().sum())
print("Teks hasil tersedia:",
      df["processed_text"].notna().sum())

Total dataset: 220051
Komentar sudah diproses: 7000
Teks hasil tersedia: 7000


In [30]:
# Simpan hasil sementara agar pekerjaan tidak hilang
df.to_csv("dataset_checkpoint.csv", index=False, encoding="utf-8-sig")

print("Checkpoint berhasil disimpan.")
print("Komentar sudah diproses:",
      df["processed_text"].notna().sum())


Checkpoint berhasil disimpan.
Komentar sudah diproses: 7000


In [31]:
checkpoint = pd.read_csv("dataset_checkpoint.csv")

print("Total baris checkpoint:", len(checkpoint))
print("Kolom hasil stemming tersedia:",
      "tokens_preprocessed" in checkpoint.columns)
print("Komentar dengan hasil teks:",
      checkpoint["processed_text"].notna().sum())


C:\Users\acer\AppData\Local\Temp\ipykernel_31924\1689687293.py:1: DtypeWarning: Columns (0: tokens_preprocessed, 1: processed_text) have mixed types. Specify dtype option on import or set low_memory=False.
  checkpoint = pd.read_csv("dataset_checkpoint.csv")


Total baris checkpoint: 220051
Kolom hasil stemming tersedia: True
Komentar dengan hasil teks: 6585


In [32]:
print("Hasil tokens_preprocessed terisi:",
      checkpoint["tokens_preprocessed"].notna().sum())

print("Hasil processed_text terisi:",
      checkpoint["processed_text"].notna().sum())

print("\nContoh hasil checkpoint:")
display(
    checkpoint.loc[
        checkpoint["processed_text"].notna(),
        ["comment", "processed_text"]
    ].head(5)
)


Hasil tokens_preprocessed terisi: 7000
Hasil processed_text terisi: 6585

Contoh hasil checkpoint:


,comment,processed_text
0,Beliau bilang untung nya 2 m ?,beliau bilang untung nya m
1,Makasih Mas,makasih mas
2,Gimana untung nya 2 m..barang yg di jual ajh ga ada 1 m 😁,gimana untung nya m barang yg jual ajh ga m
3,[Sticker] dibilang Makasih Mas udah 2M itu,sticker bilang makasih mas udah m
5,"Kalo niat membantu UMKM, harusnya koprasi merah putih dibuka dalam bentuk warehouse (gudang), bu...",kalo niat bantu umkm koprasi merah putih buka bentuk warehouse gudang bukan bentuk toko belanja ...


In [33]:
# Pulihkan hasil checkpoint ke df
df = checkpoint.copy()

# Tentukan baris yang belum memiliki hasil preprocessing
belum_diproses = df["tokens_preprocessed"].isna()

print("Total data:", len(df))
print("Baris yang belum diproses:", belum_diproses.sum())
print("Baris yang sudah memiliki hasil:", (~belum_diproses).sum())


Total data: 220051
Baris yang belum diproses: 213051
Baris yang sudah memiliki hasil: 7000


In [ ]:
#Pemrosesan Seluruh Dataset dan Penyimpanan Checkpoint
#Bagian ini memproses komentar secara bertahap dengan batch 1.000 data dan menyimpan hasil preprocessing serta checkpoint secara berkala.
# Pastikan kolom hasil dapat menyimpan list dan teks
df["tokens_preprocessed"] = df["tokens_preprocessed"].astype(object)
df["processed_text"] = df["processed_text"].astype(object)

ukuran_batch = 1000
total_data = len(df)

# Ambil indeks yang belum memiliki hasil
indeks_belum = df.index[df["tokens_preprocessed"].isna()].tolist()

for posisi in range(0, len(indeks_belum), ukuran_batch):
    indeks_batch = indeks_belum[posisi:posisi + ukuran_batch]

    hasil = (
        df.loc[indeks_batch, "comment"]
        .fillna("")
        .apply(preprocessing)
    )

    # Simpan hasil satu per satu agar list tidak dianggap sebagai indeks
    for idx, tokens in hasil.items():
        df.at[idx, "tokens_preprocessed"] = tokens
        df.at[idx, "processed_text"] = " ".join(tokens)

    # Simpan checkpoint setelah setiap batch
    df.to_csv("dataset_checkpoint.csv", index=False, encoding="utf-8-sig")

    jumlah_selesai = df["tokens_preprocessed"].notna().sum()
    print(f"Selesai memproses {jumlah_selesai} dari {total_data} komentar")

print("Proses selesai.")


Selesai memproses 8000 dari 220051 komentar
Selesai memproses 9000 dari 220051 komentar
Selesai memproses 10000 dari 220051 komentar
Selesai memproses 11000 dari 220051 komentar
Selesai memproses 12000 dari 220051 komentar
Selesai memproses 13000 dari 220051 komentar
Selesai memproses 14000 dari 220051 komentar
Selesai memproses 15000 dari 220051 komentar
Selesai memproses 16000 dari 220051 komentar
Selesai memproses 17000 dari 220051 komentar
Selesai memproses 18000 dari 220051 komentar
Selesai memproses 19000 dari 220051 komentar
Selesai memproses 20000 dari 220051 komentar
Selesai memproses 21000 dari 220051 komentar
Selesai memproses 22000 dari 220051 komentar
Selesai memproses 23000 dari 220051 komentar
Selesai memproses 24000 dari 220051 komentar
Selesai memproses 25000 dari 220051 komentar
Selesai memproses 26000 dari 220051 komentar
Selesai memproses 27000 dari 220051 komentar
Selesai memproses 28000 dari 220051 komentar
Selesai memproses 29000 dari 220051 komentar
Selesai memp

In [36]:
df.to_csv(
    "dataset_preprocessed.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Dataset final berhasil disimpan!")
print("Total komentar:", len(df))
print(
    "Komentar yang sudah diproses:",
    df["tokens_preprocessed"].notna().sum()
)


Dataset final berhasil disimpan!
Total komentar: 220051
Komentar yang sudah diproses: 220051


In [37]:
print("Total baris dataset:", len(df))
print("Komentar yang sudah diproses:",
      df["tokens_preprocessed"].notna().sum())

display(
    df[["comment", "processed_text"]].head(5)
)


Total baris dataset: 220051
Komentar yang sudah diproses: 220051


,comment,processed_text
0,Beliau bilang untung nya 2 m ?,beliau bilang untung nya m
1,Makasih Mas,makasih mas
2,Gimana untung nya 2 m..barang yg di jual ajh ga ada 1 m 😁,gimana untung nya m barang yg jual ajh ga m
3,[Sticker] dibilang Makasih Mas udah 2M itu,sticker bilang makasih mas udah m
4,😳😳😳,NaN


In [40]:
#tabel eksperimen 20 komentar

jumlah_contoh = 20

tabel_perbandingan = pd.DataFrame({
    "Raw": df["comment"].head(jumlah_contoh),
    "Lowercase": df["lowercase_text"].head(jumlah_contoh),
    "Cleaned": df["clean_text"].head(jumlah_contoh),
    "Tokenized": df["tokens"].head(jumlah_contoh),
    "Stopword Removal": df["tokens_no_stopword"].head(jumlah_contoh),
    "Stemming": df["tokens_preprocessed"].head(jumlah_contoh)
})

display(tabel_perbandingan)

,Raw,Lowercase,Cleaned,Tokenized,Stopword Removal,Stemming
0,Beliau bilang untung nya 2 m ?,beliau bilang untung nya 2 m ?,beliau bilang untung nya m,"['beliau', 'bilang', 'untung', 'nya', 'm']","['beliau', 'bilang', 'untung', 'nya', 'm']","['beliau', 'bilang', 'untung', 'nya', 'm']"
1,Makasih Mas,makasih mas,makasih mas,"['makasih', 'mas']","['makasih', 'mas']","['makasih', 'mas']"
2,Gimana untung nya 2 m..barang yg di jual ajh ga ada 1 m 😁,gimana untung nya 2 m..barang yg di jual ajh ga ada 1 m 😁,gimana untung nya m barang yg di jual ajh ga ada m,"['gimana', 'untung', 'nya', 'mbarang', 'yg', 'di', 'jual', 'ajh', 'ga', 'ada', 'm', '😁']","['gimana', 'untung', 'nya', 'mbarang', 'yg', 'jual', 'ajh', 'ga', 'm', '😁']","['gimana', 'untung', 'nya', 'm', 'barang', 'yg', 'jual', 'ajh', 'ga', 'm']"
3,[Sticker] dibilang Makasih Mas udah 2M itu,[sticker] dibilang makasih mas udah 2m itu,sticker dibilang makasih mas udah m itu,"['sticker', 'dibilang', 'makasih', 'mas', 'udah', 'm', 'itu']","['sticker', 'dibilang', 'makasih', 'mas', 'udah', 'm']","['sticker', 'bilang', 'makasih', 'mas', 'udah', 'm']"
4,😳😳😳,😳😳😳,NaN,['😳😳😳'],['😳😳😳'],[]
5,"Kalo niat membantu UMKM, harusnya koprasi merah putih dibuka dalam bentuk warehouse (gudang), bu...","kalo niat membantu umkm, harusnya koprasi merah putih dibuka dalam bentuk warehouse (gudang), bu...",kalo niat membantu umkm harusnya koprasi merah putih dibuka dalam bentuk warehouse gudang bukan ...,"['kalo', 'niat', 'membantu', 'umkm', 'harusnya', 'koprasi', 'merah', 'putih', 'dibuka', 'dalam',...","['kalo', 'niat', 'membantu', 'umkm', 'koprasi', 'merah', 'putih', 'dibuka', 'bentuk', 'warehouse...","['kalo', 'niat', 'bantu', 'umkm', 'koprasi', 'merah', 'putih', 'buka', 'bentuk', 'warehouse', 'g..."
6,sabar mbak paling 6 bulan dah tutup,sabar mbak paling 6 bulan dah tutup,sabar mbak paling bulan dah tutup,"['sabar', 'mbak', 'paling', 'bulan', 'dah', 'tutup']","['sabar', 'mbak', 'dah', 'tutup']","['sabar', 'mbak', 'dah', 'tutup']"
7,"kalo ternyata harga murah, saya harap bisa berjalan terus.","kalo ternyata harga murah, saya harap bisa berjalan terus.",kalo ternyata harga murah saya harap bisa berjalan terus,"['kalo', 'ternyata', 'harga', 'murah', 'saya', 'harap', 'bisa', 'berjalan', 'terus']","['kalo', 'harga', 'murah', 'harap', 'berjalan']","['kalo', 'harga', 'murah', 'harap', 'jalan']"
8,"SAYA SETUJU BANG, soalnya saya punya warung 😔","saya setuju bang, soalnya saya punya warung 😔",saya setuju bang soalnya saya punya warung,"['saya', 'setuju', 'bang', 'soalnya', 'saya', 'punya', 'warung', '😔']","['setuju', 'bang', 'warung', '😔']","['tuju', 'bang', 'warung']"
9,Ancen Dancuk 🥵🥵🥵,ancen dancuk 🥵🥵🥵,ancen dancuk,"['ancen', 'dancuk', '🥵🥵🥵']","['ancen', 'dancuk', '🥵🥵🥵']","['ancen', 'dancuk']"


In [39]:

import os
import pandas as pd

file_final = "dataset_preprocessed.csv"

print("File ditemukan:", os.path.exists(file_final))

if os.path.exists(file_final):
    cek = pd.read_csv(file_final, low_memory=False)
    print("Jumlah baris:", len(cek))
    print("Kolom tersedia:", cek.columns.tolist())
    display(cek[["comment", "processed_text"]].head(5))


File ditemukan: True
Jumlah baris: 220051
Kolom tersedia: ['video_id', 'comment_id', 'parent_comment_id', 'level', 'username', 'nickname', 'comment', 'create_time', 'lowercase_text', 'clean_text', 'tokens', 'tokens_no_stopword', 'tokens_preprocessed', 'processed_text']


,comment,processed_text
0,Beliau bilang untung nya 2 m ?,beliau bilang untung nya m
1,Makasih Mas,makasih mas
2,Gimana untung nya 2 m..barang yg di jual ajh ga ada 1 m 😁,gimana untung nya m barang yg jual ajh ga m
3,[Sticker] dibilang Makasih Mas udah 2M itu,sticker bilang makasih mas udah m
4,😳😳😳,NaN
